In [13]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
import json
import pickle

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
client = weaviate.connect_to_local()

/home/razor/Documents/alzheimers-rag/.venv/lib/python3.12/site-packages/weaviate/warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
/tmp/ipykernel_956509/2610504379.py:1: ResourceWarning: unclosed <socket.socket fd=75, family=2, type=1, proto=6, laddr=('127.0.0.1', 39804), raddr=('127.0.0.1', 8080)>
  client = weaviate.connect_to_local()


# DB Initialization

In [55]:
client.collections.delete("AlzheimersChunks")
client.collections.create(
    name="AlzheimersChunks",
    vectorizer_config=None,
    properties=[
        Property(
            name="pmid",
            data_type=DataType.INT,
        ),
        Property(
            name="title",
            data_type=DataType.TEXT,
        ),
        Property(
            name="text",
            data_type=DataType.TEXT,
        ),
        Property(
            name="journal",
            data_type=DataType.TEXT,
        ),
        Property(
            name="year",
            data_type=DataType.INT,
        ),
    ]
)

In [16]:
collection = client.collections.get("AlzheimersChunks")
collection

# Tokenization

In [ ]:
doc_tokenizer = AutoTokenizer.from_pretrained(
    "ncbi/MedCPT-Article-Encoder")
doc_model = AutoModel.from_pretrained("ncbi/MedCPT-Article-Encoder").to(device)
doc_model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7361.22it/s]


In [9]:
query_tokenizer = AutoTokenizer.from_pretrained(
    "ncbi/MedCPT-Query-Encoder")
query_model = AutoModel.from_pretrained("ncbi/MedCPT-Query-Encoder").to(device)
query_model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7427.77it/s]


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
from tqdm.auto import tqdm

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    return torch.sum(
        token_embeddings * input_mask_expanded,
        1
    ) / torch.clamp(
        input_mask_expanded.sum(1),
        min=1e-9
    )


def encode_documents_batch(
    texts,
    batch_size=32
):
    all_embeddings = []

    for i in tqdm(
        range(0, len(texts), batch_size),
        desc="Encoding documents",
        unit="batch"
    ):
        batch = texts[i:i + batch_size]

        encoded = doc_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = doc_model(**encoded)

        embeddings = mean_pooling(
            outputs,
            encoded["attention_mask"]
        )

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        all_embeddings.extend(
            embeddings.cpu().tolist()
        )

    return all_embeddings


def encode_query(query):

    encoded_input = query_tokenizer(
        query,
        padding=True,
        truncation=True,
        return_tensors="pt",
        max_length=512
    )

    with torch.no_grad():
        model_output = query_model(
            **encoded_input
        )

    embeddings = mean_pooling(
        model_output,
        encoded_input["attention_mask"]
    )

    embeddings = F.normalize(
        embeddings,
        p=2,
        dim=1
    )

    embeddings = embeddings.cpu().tolist()

    return embeddings[0]

In [49]:
chunked_records = []
with open("/home/razor/Documents/alzheimers-rag/data/processed/alzheimers_chunks.jsonl", "r") as f:
    for line in f:
        chunked_records.append(json.loads(line))

In [53]:
texts = [
    chunk["text"]
    for chunk in chunked_records
]

In [56]:
embeddings = encode_documents_batch(
    texts,
    batch_size=8
)

Encoding documents: 100%|██████████| 256/256 [13:10<00:00,  3.09s/batch]


In [61]:
records= []
for chunk, vector in zip(chunked_records, embeddings):
        record = {
            "chunk_id": chunk["chunk_id"],
            "vector": vector,
            "metadata": {
                key: value
                for key, value in chunk.items()
                if key != "chunk_id" and key != "text"
            },
        }
        records.append(record)

In [ ]:
with open("/home/razor/Documents/alzheimers-rag/data/processed/medcpt_embeddings.jsonl", "w") as f:
        f.write(json.dumps(records))

In [65]:
with open("/home/razor/Documents/alzheimers-rag/data/embeddings.pkl", "wb") as f:
    pickle.dump(records, f)

In [67]:
with collection.batch.dynamic() as batch:
    for chunk, vector in zip(
        chunked_records,
        embeddings
    ):
        batch.add_object(
            properties={
                "pmid":
                    chunk["pmid"],
                "title":
                    chunk["title"],
                "text":
                    chunk["text"],
                "journal":
                    chunk["journal"],
                "year":
                    int(chunk["year"])
                    if chunk["year"]
                    else 0
            },
            vector=vector
        )

In [9]:
query = """
Is beta-amyloid a protein ?
"""

query_vector = encode_query(
    query
)

In [12]:
response = collection.query.near_vector(
    near_vector=query_vector,
    limit=5,
    return_properties=[
        "title",
        "text",
        "journal",
        "year"
    ]
)

In [13]:
for obj in response.objects:

    print("=" * 80)

    print(obj.properties["title"])

    print()

    print(obj.properties["text"][:500])

The aggregation of amyloid-β: from condensation, nucleation, and conformation to targeting therapy.

This review summarizes the structural characteristics of various Aβ aggregates, along with related regulatory elements, their effects on cellular processes, and therapeutic strategies targeting Aβ and its aggregates. This overview contributes to a better understanding of the complex mechanisms underlying Aβ aggregation and its pathological consequences, as well as the basis for future therapies targeted to Aβ and aggregates.
Tau and β-Amyloid Relevant Pathology as a Central Therapeutic Target in Alzheimer's Disease.

Alzheimer's disease (AD) is the leading cause of dementia, responsible for approximately 60-70% of cases globally. AD is a gradually progressive neurodegenerative disorder that is characterized by widespread deposition of β-amyloid (Aβ) plaques, followed by aggregation of tau protein in the neocortex, neurodegeneration, and cognitive decline. Within these complex pathologic

# Evaluation

In [7]:
queries = [
    "How does amyloid-beta contribute to neurodegeneration?",
    "What biomarkers are used for early Alzheimer's detection?",
    "What evidence exists for anti-amyloid therapies?",
    "How is insulin resistance associated with Alzheimer's disease?",
    "What is the difference between tau pathology and amyloid pathology?"
]

In [20]:
def retrieve(query, top_k=5):
    query_vector = encode_query(query)
    response = collection.query.near_vector(
            near_vector=query_vector,
            limit=top_k,
            return_properties=[
                "title",
                "text",
                "journal",
                "year"
            ]
        )
    return response.objects

In [27]:
results = retrieve("What is the role of microglia in Alzheimer's disease?")

In [29]:
for i, obj in enumerate(results):
    print(f"Result {i + 1}:")
    print("Title:", obj.properties["title"])
    print("Journal:", obj.properties["journal"])
    print("Year:", obj.properties["year"])
    print("Text:", obj.properties["text"])
    print("=" * 80)

Result 1:
Title: Tau and β-Amyloid Relevant Pathology as a Central Therapeutic Target in Alzheimer's Disease.
Journal: Biomolecules
Year: 2026
Text: Alzheimer's disease (AD) is the leading cause of dementia, responsible for approximately 60-70% of cases globally. AD is a gradually progressive neurodegenerative disorder that is characterized by widespread deposition of β-amyloid (Aβ) plaques, followed by aggregation of tau protein in the neocortex, neurodegeneration, and cognitive decline. Within these complex pathological interactions, Aβ and tau proteins, together with astrogliosis, neuroinflammation, and other factors, play a key role in the development of clinical AD. Accumulating evidence indicates that the formation of protein oligomers, followed by their aggregation into pathological fibrils, constitutes an early and critical step in the pathogenesis of the disease. Specific pathological proteins are often treated as biomarkers of particular diseases because their presence, conce